In [8]:
import os
import random
from pathlib import Path
from typing import Tuple
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import pandas as pd
import numpy as np
import cv2

# -------- Config --------
num_samples = 200  # change as needed
output_dir = Path("province_only_synth")
# Gen at 2x size for better quality
# Don't fix canvas size - let it adjust based on text length with consistent margins
margin_horizontal = 100  # total left+right margin (50px each side at 2x scale)
margin_vertical = 60  # total top+bottom margin (30px each side at 2x scale)

full_provinces = [
    "กรุงเทพมหานคร","กระบี่","กาญจนบุรี","กาฬสินธุ์","กำแพงเพชร","ขอนแก่น","จันทบุรี","ฉะเชิงเทรา","ชลบุรี","ชัยนาท","ชัยภูมิ","ชุมพร","เชียงราย","เชียงใหม่","ตรัง","ตราด","ตาก","นครนายก","นครปฐม","นครพนม","นครราชสีมา","นครศรีธรรมราช","นครสวรรค์","นนทบุรี","นราธิวาส","น่าน","บึงกาฬ","บุรีรัมย์","ปทุมธานี","ประจวบคีรีขันธ์","ปราจีนบุรี","ปัตตานี","พระนครศรีอยุธยา","พังงา","พัทลุง","พิจิตร","พิษณุโลก","เพชรบุรี","เพชรบูรณ์","แพร่","พะเยา","ภูเก็ต","มหาสารคาม","มุกดาหาร","แม่ฮ่องสอน","ยโสธร","ยะลา","ร้อยเอ็ด","ระนอง","ระยอง","ราชบุรี","ลพบุรี","ลำปาง","ลำพูน","เลย","ศรีสะเกษ","สกลนคร","สงขลา","สตูล","สมุทรปราการ","สมุทรสงคราม","สมุทรสาคร","สระแก้ว","สระบุรี","สิงห์บุรี","สุโขทัย","สุพรรณบุรี","สุราษฎร์ธานี","สุรินทร์","หนองคาย","หนองบัวลำภู","อ่างทอง","อำนาจเจริญ","อุดรธานี","อุตรดิตถ์","อุทัยธานี","อุบลราชธานี","เบตง",
]

palette = {
    "group1_personal": {
        "colors": [
            (255, 255, 255), (245, 245, 245), (230, 230, 230),
            (220, 225, 230), (210, 210, 210)
        ],
        "weight": 0.8,
    },
    "group2_taxi": {
        "colors": [
            (255, 235, 100), (255, 200, 90), (255, 180, 90),
            (150, 255, 150), (190, 240, 120)
        ],
        "weight": 0.1,
    },
    "group3_graphic": {
        "colors": [
            (255, 210, 230), (200, 220, 255), (255, 220, 170),
            (210, 200, 255), (180, 200, 210)
        ],
        "weight": 0.1,
    },
}

def pick_font():
    preferred = "C:/Windows/Fonts/Sarun's ThangLuang.ttf"
    candidates = [
        preferred,
        "C:/Windows/Fonts/tahoma.ttf",
        "C:/Windows/Fonts/THSarabunNew.ttf",
        "C:/Windows/Fonts/THSarabunNew Bold.ttf",
    ]
    for path in candidates:
        if os.path.exists(path):
            try:
                return ImageFont.truetype(path, 32)  # Larger font for 2x canvas
            except OSError:
                continue
    print("Warning: Thai font not found, using default")
    return ImageFont.load_default()

font = pick_font()

def choose_palette() -> Tuple[str, Tuple[int, int, int]]:
    groups = list(palette.keys())
    weights = [palette[g]["weight"] for g in groups]
    group = random.choices(groups, weights=weights, k=1)[0]
    color = random.choice(palette[group]["colors"])
    return group, color

def draw_province(province: str, bg_color: Tuple[int, int, int]) -> Image.Image:
    # Create temporary image to measure text size
    temp_img = Image.new("RGB", (1, 1))
    temp_draw = ImageDraw.Draw(temp_img)
    bbox = temp_draw.textbbox((0, 0), province, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]
    
    # Calculate image size based on text + consistent margins (with small variation)
    # This ensures margins are proportional regardless of text length
    w = text_w + margin_horizontal + random.randint(-5, 10)
    h = text_h + margin_vertical + random.randint(-3, 5)
    
    # Create actual image
    img = Image.new("RGB", (w, h), bg_color)
    draw = ImageDraw.Draw(img)
    
    # No border
    
    # text centered using anchor='mm' (middle-middle)
    center_x = w // 2
    center_y = h // 2
    draw.text((center_x, center_y), province, fill=(50, 50, 50), font=font, anchor='mm')
    return img

def apply_homography(img: np.ndarray) -> np.ndarray:
    """Apply homography transformation to simulate perspective distortion"""
    height, width = img.shape[:2]
    
    # Define random perspective transformation
    src_points = np.float32([
        [0, 0],
        [width, 0],
        [width, height],
        [0, height]
    ])
    
    # Destination points (with random distortion)
    distortion = random.uniform(0.03, 0.08)  # ลดลงเพื่อให้อ่านง่ายขึ้น
    dst_points = np.float32([
        [random.uniform(0, width*distortion), random.uniform(0, height*distortion)],
        [width - random.uniform(0, width*distortion), random.uniform(0, height*distortion)],
        [width - random.uniform(0, width*distortion), height - random.uniform(0, height*distortion)],
        [random.uniform(0, width*distortion), height - random.uniform(0, height*distortion)]
    ])
    
    # Calculate homography matrix
    matrix = cv2.getPerspectiveTransform(src_points, dst_points)
    
    # Apply transformation
    transformed = cv2.warpPerspective(img, matrix, (width, height), 
                                     flags=cv2.INTER_CUBIC,
                                     borderMode=cv2.BORDER_REPLICATE)
    
    return transformed

def apply_augmentation(img: np.ndarray) -> np.ndarray:
    """Apply data augmentation: translation, rotation, scaling"""
    height, width = img.shape[:2]
    
    # Random rotation (-10 to 10 degrees)
    angle = random.uniform(-10, 10)
    rotation_matrix = cv2.getRotationMatrix2D((width/2, height/2), angle, 1.0)
    img = cv2.warpAffine(img, rotation_matrix, (width, height),
                        flags=cv2.INTER_CUBIC,
                        borderMode=cv2.BORDER_REPLICATE)
    
    # Random scaling (0.85 to 1.15)
    scale = random.uniform(0.85, 1.15)
    new_width = int(width * scale)
    new_height = int(height * scale)
    img = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_CUBIC)
    
    # Crop or pad to original size
    if scale > 1.0:
        start_x = (new_width - width) // 2
        start_y = (new_height - height) // 2
        img = img[start_y:start_y+height, start_x:start_x+width]
    else:
        pad_x = (width - new_width) // 2
        pad_y = (height - new_height) // 2
        img = cv2.copyMakeBorder(img, pad_y, height-new_height-pad_y, 
                                pad_x, width-new_width-pad_x, 
                                cv2.BORDER_REPLICATE)
    
    # Random translation (-20 to 20 pixels)
    tx = random.randint(-20, 20)
    ty = random.randint(-10, 10)
    translation_matrix = np.float32([[1, 0, tx], [0, 1, ty]])
    img = cv2.warpAffine(img, translation_matrix, (width, height),
                        borderMode=cv2.BORDER_REPLICATE)
    
    return img

def add_noise_and_blur(img: np.ndarray) -> np.ndarray:
    """Add random noise and blur to make image more realistic"""
    # Add Gaussian noise (force GRAYSCALE grain to avoid RGB speckles)
    if random.random() < 0.3:
        if img.ndim == 3:
            noise = np.random.normal(0, 10, (img.shape[0], img.shape[1], 1)).astype(np.int16)
        else:
            noise = np.random.normal(0, 10, img.shape).astype(np.int16)
        img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    # Add blur
    if random.random() < 1:
        kernel_size = random.choice([3, 5])
        img = cv2.GaussianBlur(img, (kernel_size, kernel_size), 0)
    
    # Adjust brightness
    if random.random() < 0.3:
        brightness = random.uniform(0.8, 1.2)
        img = cv2.convertScaleAbs(img, alpha=brightness, beta=0)
    
    return img

def apply_bicubic_interpolation(img: np.ndarray, scale: float = 0.5) -> np.ndarray:
    """Apply bicubic interpolation to adjust image sharpness"""
    height, width = img.shape[:2]
    
    # Downscale
    small = cv2.resize(img, (int(width*scale), int(height*scale)),
                      interpolation=cv2.INTER_CUBIC)
    
    # Upscale back to original size
    img = cv2.resize(small, (width, height), interpolation=cv2.INTER_CUBIC)
    
    return img

def apply_motion_blur(image, size=None):
    if size is None:
        size = random.choice([5, 7, 9, 11, 15]) # เลือกความเบลอ (ยิ่งเลขเยอะยิ่งเบลอมาก)
    
    # สร้าง Kernel สำหรับ Motion Blur
    kernel_motion_blur = np.zeros((size, size))
    
    # สุ่มทิศทางเบลอ (แนวนอน หรือ แนวตั้งเล็กน้อย)
    # ส่วนใหญ่รถวิ่งแนวนอนตัดกล้อง หรือวิ่งเข้าหากล้อง
    angle = random.randint(-10, 10) 
    
    # (วิธีลัด) ใช้ cv2.getRotationMatrix2D หมุนเส้นตรงเพื่อทำ Directional Blur ได้
    # แต่วิธีง่ายสุดคือทำแนวนอนตรงๆ (Horizontal Motion Blur) ซึ่งเจอบ่อยสุด
    kernel_motion_blur[int((size-1)/2), :] = np.ones(size)
    kernel_motion_blur = kernel_motion_blur / size
    
    # Apply Blur
    output = cv2.filter2D(image, -1, kernel_motion_blur)
    return output

def thicken_text(image):
    """ทำให้ตัวหนังสือหนาขึ้น (เหมือนหมึกซึมหรือแสงฟุ้ง)"""
    # ใช้ Erode เพราะพื้นหลังขาว ตัวหนังสือดำ -> การกัดสีขาวออก = ตัวหนังสือดำหนาขึ้น
    # Kernel ขนาด 2x2 หรือ 3x3 กำลังดี
    k_size = random.choice([2, 2]) 
    kernel = np.ones((k_size, k_size), np.uint8)
    # Iterations=1 พอครับ เดี๋ยวหนาเกินอ่านไม่ออก
    return cv2.erode(image, kernel, iterations=1)

def apply_heavy_motion_blur(image):
    """เบลอแนวนอนแบบหนักหน่วง (จำลองรถวิ่งเร็ว)"""
    # Kernel ลดลงเพื่อให้ยังอ่านออก
    kernel_size = random.choice([5, 7, 9, 11])
    
    # สร้าง Kernel แนวนอน (Horizontal Only)
    kernel = np.zeros((kernel_size, kernel_size))
    kernel[int((kernel_size-1)/2), :] = np.ones(kernel_size)
    kernel /= kernel_size
    
    return cv2.filter2D(image, -1, kernel)

def fade_image(image):
    """ทำให้ภาพดูซีด/จาง (Low Contrast)"""
    h, w, c = image.shape
    # สร้างภาพสีเทากลางๆ (Gray Overlay)
    gray_val = random.randint(100, 180) 
    gray_overlay = np.full((h, w, c), gray_val, dtype=np.uint8)
    
    # ผสมภาพเดิมกับสีเทา (Alpha คือความจาง)
    alpha = random.uniform(0.1, 0.25)  # ลดลงเพื่อไม่ให้ซีดเกินไป
    faded = cv2.addWeighted(image, 1-alpha, gray_overlay, alpha, 0)
    return faded

def apply_squash(img: np.ndarray) -> np.ndarray:
    """บีบภาพแนวตั้งให้แบนลง (จำลองมุมกล้อง CCTV ที่กดลงมา)"""
    h, w = img.shape[:2]
    
    # สุ่มบีบความสูงลงเหลือ 55% - 80% ของความสูงเดิม
    # ค่านี้สำคัญมาก! Real Data ของคุณดูเตี้ยประมาณ 0.6-0.7 เลย
    squash_factor = random.uniform(0.55, 0.80) 
    
    new_h = int(h * squash_factor)
    
    # Resize เฉพาะแกน Y (ความสูง) ให้ลดลง
    img = cv2.resize(img, (w, new_h), interpolation=cv2.INTER_AREA)
    
    return img

def ensure_dirs():
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "images").mkdir(exist_ok=True)

ensure_dirs()

rows = []

for i in range(num_samples):
    province = random.choice(full_provinces)
    group, color = choose_palette()
    img_pil = draw_province(province, color)
    
    # Convert to numpy array for augmentation
    img_np = np.array(img_pil)

    # --- PHASE 0: Squash (บีบให้เตี้ยก่อนเลย!) ---
    # ใส่ 100% เลยครับ เพราะกล้องคุณเป็นมุมสูงทุกรูป
    img_np = apply_squash(img_np)
    
    # --- PHASE 1: Geometry (บิดรูปร่างก่อน) ---
    if random.random() < 0.5:  # 50% Homography (ลดลง)
        img_np = apply_homography(img_np)
    
    if random.random() < 0.6:  # 60% Augmentation (ลดลง)
        img_np = apply_augmentation(img_np)

    # --- PHASE 2: Structure (ทำให้เส้นหนา) ---
    # *ใส่ก่อน Blur* เพื่อให้ส่วนที่หนาขึ้นถูกเบลอไปด้วยกันดูเป็นธรรมชาติ
    if random.random() < 0.3:  # 30% ตัวหนังสือบวม (ลดลงมาก)
        img_np = thicken_text(img_np)

    # --- PHASE 3: Motion (เบลอจากการเคลื่อนที่) ---
    # ลดลงเพื่อให้อ่านออก
    if random.random() < 0.8:  # 80% Heavy Blur (ลดจาก 85%)
        img_np = apply_heavy_motion_blur(img_np)
    elif random.random() < 0.3:  # 30% เบลอเบาๆ
        img_np = add_noise_and_blur(img_np)

    # --- PHASE 4: Lighting/Sensor (แสงและสี) ---
    # *ใส่หลัง Blur* เพราะ Sensor Noise เกิดขึ้นบนภาพที่เบลอแล้ว
    if random.random() < 0.6:  # 60% ภาพซีด/จาง (ลดจาก 70%)
        img_np = fade_image(img_np)

    if random.random() < 0.25:  # 25% Noise เพิ่มเติม (ลดลง)
        img_np = add_noise_and_blur(img_np)

    # --- PHASE 5: Resolution (ขั้นตอนสุดท้าย) ---
    # ย่อขยายเพื่อลดคุณภาพความคมชัดปิดท้าย
    if random.random() < 0.3:  # 30% bicubic (ลดลง)
        img_np = apply_bicubic_interpolation(img_np, scale=random.uniform(0.5, 0.8))

    # --- PHASE 6: Final Resize (Ready to Train) ---
    # ย่อให้เหลือขนาดที่โมเดลต้องการเลย (แนะนำ 128x32 สำหรับป้ายทะเบียนยาว)
    target_w, target_h = 128, 32
    img_np = cv2.resize(img_np, (target_w, target_h), interpolation=cv2.INTER_AREA)
    
    # Save image
    fname_only = f"synth_province_{i:05d}_lower.jpg"
    fname = f"images/{fname_only}"
    cv2.imwrite(str(output_dir / fname), cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))
    province_code = f"TH-{full_provinces.index(province) + 1:02d}"
    
    # Simplified CSV format
    rows.append({
        "filename": fname_only,
        "label": province,
        "source": "synthetic",
        "province_code": province_code,
        "province_description": province
    })

df = pd.DataFrame(rows)
csv_path = output_dir / "labels.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"Generated {len(df)} samples -> {csv_path}")
print(f"Image size: text_width + {margin_horizontal}px, text_height + {margin_vertical}px (consistent margins)")
print("Augmentations applied: homography, rotation, scaling, translation, noise, blur, bicubic, motion blur")
df.head()

Generated 200 samples -> province_only_synth\labels.csv
Image size: text_width + 100px, text_height + 60px (consistent margins)
Augmentations applied: homography, rotation, scaling, translation, noise, blur, bicubic, motion blur


,filename,label,source,province_code,province_description
0,synth_province_00000_lower.jpg,อ่างทอง,synthetic,TH-72,อ่างทอง
1,synth_province_00001_lower.jpg,ยโสธร,synthetic,TH-46,ยโสธร
2,synth_province_00002_lower.jpg,สกลนคร,synthetic,TH-57,สกลนคร
3,synth_province_00003_lower.jpg,สมุทรสงคราม,synthetic,TH-61,สมุทรสงคราม
4,synth_province_00004_lower.jpg,นครพนม,synthetic,TH-20,นครพนม
